In [ ]:
!pip install transformers datasets torch librosa

In [ ]:
import torch
import numpy as np

from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
from datasets import load_dataset, Audio
from torch.utils.data import DataLoader

In [ ]:
#Φόρτωση fine-tuned μοντέλου και feature extractor
repo_id = "VasilisAsim/data2vec-finetuned-IEMOCAP-categorical"
model_ft = AutoModelForAudioClassification.from_pretrained(repo_id)
feature_extractor = AutoFeatureExtractor.from_pretrained(repo_id)

device = "cuda" if torch.cuda.is_available() else "cpu"
model_ft = model_ft.to(device)
model_ft.eval()

In [ ]:
label2id = model_ft.config.label2id
id2label = model_ft.config.id2label

In [ ]:
from datasets import load_dataset, Audio, DatasetDict
from sklearn.model_selection import GroupShuffleSplit
import numpy as np

dataset = load_dataset("AbstractTTS/IEMOCAP", split="train")
dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))

def merge_and_filter(batch):
    emo = batch["major_emotion"]
    if emo == "excited":
        emo = "happy"
    batch["emotion_4class"] = emo
    return batch

dataset = dataset.map(merge_and_filter)
dataset = dataset.filter(lambda b: b["emotion_4class"] in ["angry", "happy", "neutral", "sad"])

def add_speaker(batch):
    parts = batch["file"].split("_")
    session = parts[0][:5]
    speaker_gender = parts[-1][0]
    batch["speaker_id"] = f"{session}_{speaker_gender}"
    return batch

dataset = dataset.map(add_speaker)

labels = sorted(set(dataset["emotion_4class"]))
num_labels = len(labels)
label2id = {name: i for i, name in enumerate(labels)}
id2label = {i: name for i, name in enumerate(labels)}
print("Κλάσεις:", labels)

splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(X=range(len(dataset)), groups=dataset["speaker_id"]))

dataset = DatasetDict({
    "train": dataset.select(train_idx),
    "test": dataset.select(test_idx),
})

In [ ]:
MAX_SAMPLES = 6 * 16_000

def is_valid_audio(batch):
    audio = batch["audio"]["array"][:MAX_SAMPLES]
    if len(audio) == 0:
        return False
    return np.std(audio) >= 1e-4

print("Πριν:", len(dataset["train"]), len(dataset["test"]))
dataset["train"] = dataset["train"].filter(is_valid_audio, load_from_cache_file=False)
dataset["test"] = dataset["test"].filter(is_valid_audio, load_from_cache_file=False)
print("Μετά:", len(dataset["train"]), len(dataset["test"]))

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]
    audio_array = audio["array"][:MAX_SAMPLES]
    batch["input_values"] = feature_extractor(audio_array, sampling_rate=audio["sampling_rate"]).input_values[0]
    batch["labels"] = label2id[batch["emotion_4class"]]
    return batch

encoded_dataset = dataset.map(prepare_dataset, remove_columns=dataset["train"].column_names, load_from_cache_file=False)

In [ ]:
def collate_fn(features):
    input_features = [{"input_values": f["input_values"]} for f in features]
    batch = feature_extractor.pad(input_features, padding=True, return_tensors="pt")
    batch["labels"] = torch.tensor([f["labels"] for f in features], dtype=torch.long)
    return batch

test_loader = DataLoader(encoded_dataset["test"], batch_size=8, collate_fn=collate_fn)

In [ ]:
all_logits, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_values = batch["input_values"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]

        #Πέρασμα κάθε στοιχείου από το finetuned μοντέλο και εξαγωγή των αποτελεσμάτων
        outputs = model_ft(input_values=input_values, attention_mask=attention_mask)

        #Αποθήκευση αποτελεσμάτων και label
        all_logits.append(outputs.logits.cpu().numpy())
        all_labels.append(labels.numpy())

#Αποθήκευση σε ενιαίο array
logits = np.concatenate(all_logits, axis=0)
labels = np.concatenate(all_labels, axis=0)

In [ ]:
from scipy.special import softmax

probs = softmax(logits, axis=1)
predictions = np.argmax(probs, axis=1)
confidences = np.max(probs, axis=1)
correct = (predictions == labels)

num_classes = probs.shape[1]
labels_onehot = np.eye(num_classes)[labels]

#Negative Log-Likelihood
def negative_log_likelihood(probs, labels, eps=1e-12):
    p_true = np.clip(probs[np.arange(len(labels)), labels], eps, 1.0)
    return -np.mean(np.log(p_true))

nll = negative_log_likelihood(probs, labels)


#Brier Score (multiclass)
def brier_score(probs, labels_onehot):
    return np.mean(np.sum((probs - labels_onehot) ** 2, axis=1))

brier = brier_score(probs, labels_onehot)


#Expected Calibration Error (equal-width bins)
def expected_calibration_error(confidences, correct, n_bins=15):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_info = []
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        # τελευταίο bin κλειστό και στα δύο άκρα
        in_bin = (confidences > lo) & (confidences <= hi) if i > 0 else (confidences >= lo) & (confidences <= hi)
        prop = np.mean(in_bin)
        if prop > 0:
            acc_bin = np.mean(correct[in_bin])
            conf_bin = np.mean(confidences[in_bin])
            ece += prop * abs(acc_bin - conf_bin)
            bin_info.append((lo, hi, acc_bin, conf_bin, prop))
    return ece, bin_info

ece, ece_bins = expected_calibration_error(confidences, correct, n_bins=15)


#Adaptive Calibration Error (equal-frequency bins)
def adaptive_calibration_error(confidences, correct, n_bins=15):
    order = np.argsort(confidences)
    conf_sorted = confidences[order]
    correct_sorted = correct[order]
    splits = np.array_split(np.arange(len(conf_sorted)), n_bins)

    ace = 0.0
    bin_info = []
    for idx in splits:
        if len(idx) == 0:
            continue
        acc_bin = np.mean(correct_sorted[idx])
        conf_bin = np.mean(conf_sorted[idx])
        prop = len(idx) / len(conf_sorted)
        ace += prop * abs(acc_bin - conf_bin)
        bin_info.append((conf_bin, acc_bin, prop))
    return ace, bin_info

ace, ace_bins = adaptive_calibration_error(confidences, correct, n_bins=15)


In [ ]:
import matplotlib.pyplot as plt

#Reliability diagram (χρησιμοποιεί τα bins του ECE)
def plot_reliability_diagram(bin_info, title="Reliability Diagram"):
    confs = [b[3] for b in bin_info]
    accs = [b[2] for b in bin_info]
    plt.figure(figsize=(5, 5))
    plt.bar(confs, accs, width=0.05, edgecolor="black", alpha=0.7, label="Model")
    plt.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.legend()
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.show()

plot_reliability_diagram(ece_bins, title=f"Reliability Diagram (ECE={ece:.4f})")


#Predictive Entropy
def predictive_entropy(probs, eps=1e-12):
    return -np.sum(probs * np.log(probs + eps), axis=1)

entropy = predictive_entropy(probs)

plt.figure(figsize=(6, 4))
plt.hist(entropy[correct], bins=30, alpha=0.6, label="Σωστές προβλέψεις")
plt.hist(entropy[~correct], bins=30, alpha=0.6, label="Λάθος προβλέψεις")
plt.xlabel("Predictive Entropy")
plt.ylabel("Πλήθος δειγμάτων")
plt.title("Κατανομή Predictive Entropy")
plt.legend()
plt.show()


#Confidence distribution: σωστές vs λάθος
plt.figure(figsize=(6, 4))
plt.hist(confidences[correct], bins=30, alpha=0.6, label="Σωστές προβλέψεις")
plt.hist(confidences[~correct], bins=30, alpha=0.6, label="Λάθος προβλέψεις")
plt.xlabel("Confidence (max probability)")
plt.ylabel("Πλήθος δειγμάτων")
plt.title("Κατανομή Confidence: Σωστά vs Λάθος")
plt.legend()
plt.show()


#Σύνοψη
print(f"NLL: {nll:.4f}")
print(f"Brier Score: {brier:.4f}")
print(f"ECE: {ece:.4f}")
print(f"ACE: {ace:.4f}")